In [14]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

In [15]:
df = pd.read_csv('../data/cohort/cohort_analysis.csv',
                 parse_dates = ["acquisition_date", 'cancellation_month'])
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-03-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-09-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,NaT,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [16]:
df.shape

(12000, 12)

In [17]:
df.info

<bound method DataFrame.info of        user_id acquisition_date cancellation_month  gender marital_status  \
0            1       2024-07-01         2025-03-01    Male        Married   
1            2       2024-04-01         2024-09-01    Male         Single   
2            3       2024-05-01                NaT    Male         Single   
3            4       2024-07-01                NaT    Male        Married   
4            5       2024-03-01         2024-04-01    Male         Single   
...        ...              ...                ...     ...            ...   
11995    11996       2024-02-01         2024-10-01  Female        Married   
11996    11997       2024-06-01                NaT  Female         Single   
11997    11998       2024-01-01         2024-11-01  Female         Single   
11998    11999       2024-01-01         2025-01-01    Male         Single   
11999    12000       2024-03-01         2024-04-01  Female        Married   

       age income_segment         country  

In [18]:
df.isnull().sum()

user_id                  0
acquisition_date         0
cancellation_month    3581
gender                   0
marital_status           0
age                      0
income_segment         631
country                  0
channel                  0
campaign_id              0
device_type              0
plan_type                0
dtype: int64

In [19]:
df.describe(include = "all")

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
count,12000.00000,12000,8419,12000,12000,12000.000000,11369,12000,12000,12000,12000,12000
unique,NaN,NaN,NaN,2,2,NaN,4,12,3,9,3,3
top,NaN,NaN,NaN,Female,Single,NaN,High,Netherlands,Paid Ads,Paid Ads_A,Android,Standard
freq,NaN,NaN,NaN,6016,7087,NaN,4823,1057,4854,1636,6582,5979
mean,6000.50000,2024-04-15 20:46:48,2024-10-30 17:22:19.684048,NaN,NaN,34.630250,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00000,2024-01-01 00:00:00,2024-04-01 00:00:00,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,3000.75000,2024-02-01 00:00:00,2024-08-01 00:00:00,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,6000.50000,2024-04-01 00:00:00,2024-11-01 00:00:00,NaN,NaN,34.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,9000.25000,2024-06-01 00:00:00,2025-01-01 00:00:00,NaN,NaN,41.000000,NaN,NaN,NaN,NaN,NaN,NaN
max,12000.00000,2024-08-01 00:00:00,2025-12-01 00:00:00,NaN,NaN,65.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Data Preperation | Cohort Month

df["acquisition_month"] = df["acquisition_date"].dt.to_period("M")

df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


In [21]:
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type,acquisition_month
0,1,2024-07-01,2025-03-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard,2024-07
1,2,2024-04-01,2024-09-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard,2024-04
2,3,2024-05-01,NaT,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard,2024-05
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard,2024-07
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic,2024-03


In [22]:
# Data Preperation | Churn Flag

df["is_churned"] = df["cancellation_month"].notna().astype(int)
df["is_churned"].value_counts()


is_churned
1    8419
0    3581
Name: count, dtype: int64

In [23]:
# Data Preperation | Calculate the Month of Churn

df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
df["cancellation_month"] = pd.to_datetime(df["cancellation_month"], errors="coerce")

df["month_churned"] = np.where(
    df["cancellation_month"].notna(),
    (
        (df["cancellation_month"].dt.year - df["acquisition_date"].dt.year) * 12
        + (df["cancellation_month"].dt.month - df["acquisition_date"].dt.month)
    ),
    np.nan
)

df[["user_id", "acquisition_date", "acquisition_month","cancellation_month", "month_churned"]].head(10)

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-03-01,8.0
1,2,2024-04-01,2024-04,2024-09-01,5.0
2,3,2024-05-01,2024-05,NaT,NaN
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0
5,6,2024-08-01,2024-08,NaT,NaN
6,7,2024-05-01,2024-05,2024-10-01,5.0
7,8,2024-05-01,2024-05,2024-11-01,6.0
8,9,2024-07-01,2024-07,NaT,NaN
9,10,2024-02-01,2024-02,2025-01-01,11.0


In [24]:
# Keep Only Valid Churn Months

df["month_churned"] = df["month_churned"].where(
    df["month_churned"].between(1, 12),
    np.nan
)

df["month_churned"].value_counts(dropna=False).sort_index()

month_churned
1.0      428
2.0      411
3.0      625
4.0      804
5.0      957
6.0     1029
7.0     1043
8.0      899
9.0      791
10.0     653
11.0     387
12.0     229
NaN     3744
Name: count, dtype: int64

In [25]:
# Cohort Size

cohort_size = (
    df.groupby("acquisition_month")
      .agg(new_users=("user_id", "count"))
      .reset_index()
)

cohort_size

,acquisition_month,new_users
0,2024-01,1502
1,2024-02,1528
2,2024-03,1465
3,2024-04,1509
4,2024-05,1541
5,2024-06,1493
6,2024-07,1500
7,2024-08,1462


In [26]:
# Count Actual Churn Events by Cohort and Tenure Month

cohort_churn = (
    df[df["month_churned"].notna()]
    .groupby(["acquisition_month", "month_churned"])
    .agg(users=("user_id", "count"))
    .reset_index()
)

cohort_churn["month_churned"] = cohort_churn["month_churned"].astype(int)

cohort_churn.head(15)

,acquisition_month,month_churned,users
0,2024-01,4,4
1,2024-01,5,7
2,2024-01,6,37
3,2024-01,7,70
4,2024-01,8,125
5,2024-01,9,186
6,2024-01,10,200
7,2024-01,11,179
8,2024-01,12,125
9,2024-02,2,2


In [27]:
# Full 12-Month Grid

max_month = 12
tenure_range = np.arange(1, max_month + 1)

full_index = pd.MultiIndex.from_product(
    [cohort_size["acquisition_month"].unique(), tenure_range],
    names=["acquisition_month", "month_churned"]
)

cohort_data = (
    cohort_churn
    .set_index(["acquisition_month", "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

cohort_data.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,0
1,2024-01,2,0
2,2024-01,3,0
3,2024-01,4,4
4,2024-01,5,7
5,2024-01,6,37
6,2024-01,7,70
7,2024-01,8,125
8,2024-01,9,186
9,2024-01,10,200


In [28]:
# Merge Cohort Size

cohort_data = cohort_data.merge(cohort_size, on="acquisition_month", how="left")
cohort_data.head(15)

,acquisition_month,month_churned,users,new_users
0,2024-01,1,0,1502
1,2024-01,2,0,1502
2,2024-01,3,0,1502
3,2024-01,4,4,1502
4,2024-01,5,7,1502
5,2024-01,6,37,1502
6,2024-01,7,70,1502
7,2024-01,8,125,1502
8,2024-01,9,186,1502
9,2024-01,10,200,1502


In [29]:
# Calculate Metrics

cohort_data = cohort_data.sort_values(["acquisition_month", "month_churned"])

cohort_data["cumulative_churn"] = (
    cohort_data.groupby("acquisition_month")["users"].cumsum()
)

cohort_data["active_users"] = (
    cohort_data["new_users"] - cohort_data["cumulative_churn"]
)

cohort_data["retention_rate"] = (
    cohort_data["active_users"] / cohort_data["new_users"]
)

cohort_data["acquisition_month"] = cohort_data["acquisition_month"].astype(str)

cohort_data.head(15)

,acquisition_month,month_churned,users,new_users,cumulative_churn,active_users,retention_rate
0,2024-01,1,0,1502,0,1502,1.000000
1,2024-01,2,0,1502,0,1502,1.000000
2,2024-01,3,0,1502,0,1502,1.000000
3,2024-01,4,4,1502,4,1498,0.997337
4,2024-01,5,7,1502,11,1491,0.992676
5,2024-01,6,37,1502,48,1454,0.968043
6,2024-01,7,70,1502,118,1384,0.921438
7,2024-01,8,125,1502,243,1259,0.838216
8,2024-01,9,186,1502,429,1073,0.714381
9,2024-01,10,200,1502,629,873,0.581225


In [ ]:
# At this point:

# users shows how many users churned in a given tenure month,
# cumulative_churn shows how many users have churned up to that point,
# active_users shows how many users remain,
# and retention_rate shows the share of the original cohort still active.

## Cohort Retention Heatmap

In [30]:
# Step 1: Prepare the Heatmap Table

heatmap_data = cohort_data.pivot(
    index="acquisition_month",
    columns="month_churned",
    values="retention_rate",
)

heatmap_data

month_churned,1,2,3,4,5,6,7,8,9,10,11,12
acquisition_month,,,,,,,,,,,,
2024-01,1.000000,1.000000,1.000000,0.997337,0.992676,0.968043,0.921438,0.838216,0.714381,0.581225,0.462051,0.378828
2024-02,1.000000,0.998691,0.994110,0.973168,0.926047,0.842932,0.716623,0.581806,0.453534,0.367801,0.327225,0.311518
2024-03,0.834130,0.717406,0.590444,0.452560,0.362457,0.312628,0.288055,0.285324,0.285324,0.284642,0.283276,0.283276
2024-04,0.918489,0.836978,0.705765,0.570577,0.455268,0.370444,0.318091,0.299536,0.292247,0.290921,0.290921,0.290921
2024-05,0.990266,0.968202,0.920182,0.846204,0.720311,0.577547,0.445814,0.363400,0.321220,0.303050,0.297859,0.295912
2024-06,0.969859,0.920965,0.841929,0.726725,0.580040,0.449431,0.373074,0.334226,0.313463,0.306095,0.305425,0.304086
2024-07,0.998667,0.993333,0.969333,0.928000,0.840000,0.717333,0.571333,0.444667,0.358000,0.308667,0.289333,0.281333
2024-08,1.000000,1.000000,0.995896,0.986320,0.967852,0.923393,0.835157,0.725034,0.607387,0.466484,0.393981,0.350889


In [31]:
import sys
print(sys.executable)

c:\Users\Tatev\miniconda3new\envs\myenv\python.exe


In [32]:
!{sys.executable} -m pip install nbformat

In [33]:
# Create the Heatmap

fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Blues",
    text_auto=".1%",
    labels={
        "x": "Months Since Acquisition",
        "y": "Acquisition Month",
        "color": "Retention Rate",
    },
    title="Cohort Retention Heatmap"
)

fig.update_layout(
    xaxis_title="Months Since Acquisition",
    yaxis_title="Acquisition Month"
)

fig.show()